# ArkIve — Fine-tuning do e5 para o RAG

Ensina o `multilingual-e5-small` a ligar relatos de consulta em PT-BR às fichas em inglês do índice RAG.

| Item | Escolha |
|---|---|
| Base | `intfloat/multilingual-e5-small` (PyTorch; o Xenova usado no backend é o mesmo modelo em ONNX) |
| Dados | relatos sintéticos PT gerados pelo Groq a partir dos chunks de `rag/index/chunks.jsonl` |
| Loss | `MultipleNegativesRankingLoss` com triplas (relato, chunk positivo, negativo difícil) |
| Saída | `onnx/model_quantized.onnx` + `tokenizer.json`, no mesmo layout do Xenova |

## Como usar
1. Colab com GPU (T4). Em *Secrets*, cadastre `GROQ_API_KEY` (opcional se o `.env` do repositório já tiver) e `HF_TOKEN` (só para publicar).
2. Rode tudo. A célula 6 grava `pares.jsonl` em `/content/arkive_finetune` a cada lote: se a cota do Groq acabar, rode a célula de novo e ela continua de onde parou. Nada é salvo fora da sessão; para retomar em outra sessão, baixe `pares.jsonl` antes e suba de volta para essa pasta.
3. A célula 11 compara com `rag/eval/baseline.json`. **Só troque o modelo do backend se os critérios passarem.**
4. Baixe `arkive-e5-vet-onnx.zip` (célula 12) ou publique no HF Hub (célula 13).

## Limitações conhecidas
- Dado sintético herda os erros do professor: revise a amostra da célula 7.
- `RAG_MIN_SCORE` foi calibrado para o modelo base; com o modelo novo a distribuição de scores muda. A célula 11 mostra a varredura do corte para recalibrar.


## 1. Instalação

In [ ]:
!pip -q install sentence-transformers "optimum[onnxruntime]" groq faiss-cpu python-dotenv

## 2. Repositório e modelo do backend

In [ ]:
import os, sys
from google.colab import userdata

SAIDA = "/content/arkive_finetune"
os.makedirs(SAIDA, exist_ok=True)

REPO = "/content/arkive_clinical_engine"
if not os.path.exists(REPO):
    !git clone -q -b feature/rag https://github.com/2TDSPO-ArkIve/arkive_clinical_engine.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)

# secret do Colab tem prioridade sobre o .env (load_dotenv não sobrescreve)
for chave in ["GROQ_API_KEY"]:
    try:
        os.environ[chave] = userdata.get(chave)
    except Exception:
        pass

# tokenizer podado + ONNX do backend: o RagRetriever precisa deles para montar
!python -m rag.retriever --download

## 3. Configuração

In [ ]:
import difflib, json, random, re, time
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

from config import GROQ_API_KEY, GROQ_MODEL_PRIMARY
from rag.eval import avaliar
from rag.retriever import RagRetriever, _norm, categorias_da_especie, expandir

BASE = "intfloat/multilingual-e5-small"
SEED = 42
MAX_POR_DOC = 5  # chunks por (fonte, titulo)
# seções com sinais, epidemiologia e hospedeiros; testes de laboratório não descrevem consulta
SECOES = re.compile(r"summary|introduc|clinic|sign|symptom|diagnos|epidemiol|transmiss|host|species affected"
                    r"|susceptib|lesion|patholog|post mortem|morbidity|key points|disease|cause|etiolog|aetiolog", re.I)
SECOES_FORA = re.compile(r"serolog|feedback|test", re.I)
LOTE_GROQ = 5  # chunks por chamada
PARES = Path(SAIDA) / "pares.jsonl"
MODELO_DIR = Path(SAIDA) / "arkive-e5-vet"
ONNX_DIR = Path(SAIDA) / "arkive-e5-vet-onnx"
HUB_REPO = ""  # ex.: "<usuario>/arkive-e5-vet"; vazio = não publica

## 4. Retriever com modelo PyTorch e baseline do e5 base

Mesmo `RagRetriever` do backend (glossário, filtro de espécie, penalidade, agregação por documento), com o modelo e o índice trocados.
O e5 base em fp32 precisa dar métricas próximas do `baseline.json` (int8). Se não der, o harness está errado: pare aqui.

In [ ]:
class STEmbedder:
    """Mesma interface do Embedder ONNX de rag/retriever.py, com SentenceTransformer."""

    def __init__(self, model):
        self.model = model

    def embed(self, textos):
        return self.model.encode(textos, normalize_embeddings=True, convert_to_numpy=True).astype("float32")


def passage(c):
    # idêntico ao build_index.ipynb e a rag/retriever.py
    return f"passage: {c['titulo']} — {c['secao']}\n{c['texto']}"


rag = RagRetriever()
CHUNKS = list(rag.chunks.values())
POR_ID = rag.chunks


def indexar(model):
    vet = model.encode([passage(c) for c in CHUNKS], batch_size=128, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype("float32")
    index = faiss.IndexIDMap2(faiss.IndexFlatIP(vet.shape[1]))
    index.add_with_ids(vet, np.array([c["id"] for c in CHUNKS], dtype=np.int64))
    return index


def avaliar_modelo(model):
    rag.modelo, rag.index = STEmbedder(model), indexar(model)
    return avaliar.main(rag)


base = SentenceTransformer(BASE, device="cuda")
m_base = avaliar_modelo(base)
INDEX_BASE = rag.index  # reaproveitado na mineração de negativos

## 5. Amostragem de chunks

In [ ]:
por_doc = {}
for c in CHUNKS:
    if SECOES.search(c["secao"]) and not SECOES_FORA.search(c["secao"]):
        por_doc.setdefault((c["fonte"], c["titulo"]), []).append(c)
rng = random.Random(SEED)
AMOSTRA = [c for cs in por_doc.values() for c in rng.sample(cs, min(MAX_POR_DOC, len(cs)))]
rng.shuffle(AMOSTRA)
print(f"{len(AMOSTRA)} chunks de {len(por_doc)} documentos -> ~{len(AMOSTRA) // LOTE_GROQ} chamadas ao Groq")

## 6. Relatos sintéticos (professor = Groq)

Checkpoint em `pares.jsonl`: rodar de novo pula os chunks já feitos.

In [ ]:
from groq import BadRequestError, Groq, RateLimitError

PROMPT = """Você gera dados de treino para um buscador de fichas veterinárias.
Para CADA trecho recebido, escreva 2 relatos de consulta em português do Brasil, como a transcrição do que é dito na consulta.
Regras:
- espécie e idade coerentes com a epidemiologia do trecho (espécie afetada, faixa etária, exposição);
- descreva sinais clínicos, histórico e exposição; NUNCA cite o nome da doença, do agente, do parasita ou do vírus;
- relato 1: tutor leigo, linguagem simples; relato 2: veterinário, termos técnicos;
- 2 a 5 frases cada.
Responda só JSON: {"relatos": [{"chunk_id": <id do trecho>, "especie": "<espécie em português, ex.: Cão, Gato, Bovino, Ave>", "relato": "..."}]}"""

cliente = Groq(api_key=GROQ_API_KEY)


def gerar(lote):
    trechos = "\n\n".join(f"[chunk_id={c['id']}] {c['titulo']} — {c['secao']}\n{c['texto']}" for c in lote)
    for tentativa in range(8):
        try:
            r = cliente.chat.completions.create(
                model=GROQ_MODEL_PRIMARY, temperature=0.8, reasoning_effort="low",
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": PROMPT}, {"role": "user", "content": trechos}])
            return json.loads(r.choices[0].message.content)["relatos"]
        except RateLimitError:
            time.sleep(min(5 * 2 ** tentativa, 300))
        except (BadRequestError, json.JSONDecodeError, KeyError) as e:
            print(f"\nlote descartado ({type(e).__name__}); volta na próxima execução")
            return []
    raise RuntimeError("cota do Groq esgotada: rode esta célula de novo mais tarde (retoma do checkpoint)")


validos = {c["id"] for c in AMOSTRA}
feitos = {json.loads(l)["chunk_id"] for l in PARES.open(encoding="utf-8")} if PARES.exists() else set()
pendentes = [c for c in AMOSTRA if c["id"] not in feitos]
for i in range(0, len(pendentes), LOTE_GROQ):
    with PARES.open("a", encoding="utf-8") as f:
        for p in gerar(pendentes[i:i + LOTE_GROQ]):
            try:
                cid = int(p["chunk_id"])
            except (KeyError, TypeError, ValueError):
                continue
            if cid in validos and p.get("relato"):
                f.write(json.dumps({"chunk_id": cid, "especie": p.get("especie", ""), "relato": p["relato"]},
                                   ensure_ascii=False) + "\n")
                feitos.add(cid)
    print(f"\r{len(feitos)}/{len(AMOSTRA)} chunks", end="")

## 7. Filtro de qualidade

Descarta relatos curtos ou que vazam o diagnóstico (título ou palavras do título com mais de 4 letras). **Revise a amostra impressa.**

In [ ]:
def vaza(p):
    titulo = _norm(POR_ID[p["chunk_id"]]["titulo"])
    relato = _norm(p["relato"])
    return any(t in relato for t in re.findall(r"[a-z]{5,}", titulo))


pares = [json.loads(l) for l in PARES.open(encoding="utf-8")]
bons = [p for p in pares if len(p["relato"]) >= 80 and not vaza(p)]
print(f"{len(pares)} relatos -> {len(bons)} após o filtro\n")
for p in random.Random(SEED).sample(bons, min(50, len(bons))):
    print(f"[{POR_ID[p['chunk_id']]['titulo']}] ({p['especie']}) {p['relato']}\n")

## 8. Split 90/10 por documento

Relatos de uma mesma ficha não caem em treino e validação ao mesmo tempo.

In [ ]:
titulos = sorted({POR_ID[p["chunk_id"]]["titulo"] for p in bons})
random.Random(SEED).shuffle(titulos)
VAL = set(titulos[: len(titulos) // 10])
treino = [p for p in bons if POR_ID[p["chunk_id"]]["titulo"] not in VAL]
val = [p for p in bons if POR_ID[p["chunk_id"]]["titulo"] in VAL]
print(f"treino: {len(treino)} relatos | validação: {len(val)} relatos de {len(VAL)} documentos")

## 9. Negativos difíceis

Top-20 do e5 base, sem: o mesmo documento, a mesma doença em outra fonte (título parecido) e candidatos com score ≥ 98% do positivo (provável falso negativo).
Quando existe, prefere um documento de espécie incompatível com o relato (o caso do Bob).
A consulta passa por `expandir()` como no backend.

In [ ]:
def query(p):
    return f"query: {expandir(p['relato'])}"


def parecido(a, b):
    return difflib.SequenceMatcher(None, _norm(a), _norm(b)).ratio() > 0.8


q = base.encode([query(p) for p in treino], batch_size=128, normalize_embeddings=True,
                convert_to_numpy=True, show_progress_bar=True).astype("float32")
S, I = INDEX_BASE.search(q, 20)
triplas = []
for p, v, ss, ii in zip(treino, q, S, I):
    pos = POR_ID[p["chunk_id"]]
    s_pos = float(v @ INDEX_BASE.reconstruct(pos["id"]))
    cands = [POR_ID[int(i)] for s, i in zip(ss, ii)
             if i != -1 and s < 0.98 * s_pos
             and POR_ID[int(i)]["titulo"] != pos["titulo"] and not parecido(POR_ID[int(i)]["titulo"], pos["titulo"])]
    if not cands:
        continue
    permitidas = categorias_da_especie(p["especie"])
    errada = [c for c in cands if permitidas is not None and c["categoria"] not in permitidas]
    triplas.append({"anchor": query(p), "positive": passage(pos), "negative": passage((errada or cands)[0])})
print(f"{len(triplas)} triplas de {len(treino)} relatos")

## 10. Treino

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.training_args import BatchSamplers

# validação: acertar qualquer chunk do documento certo
ids_por_titulo = {}
for c in CHUNKS:
    ids_por_titulo.setdefault(c["titulo"], set()).add(str(c["id"]))
ir = InformationRetrievalEvaluator(
    queries={str(n): query(p) for n, p in enumerate(val)},
    corpus={str(c["id"]): passage(c) for c in CHUNKS},
    relevant_docs={str(n): ids_por_titulo[POR_ID[p["chunk_id"]]["titulo"]] for n, p in enumerate(val)},
    name="val", accuracy_at_k=[3, 10], mrr_at_k=[10], show_progress_bar=True)
print("e5 base:", ir(base))

model = SentenceTransformer(BASE, device="cuda")
args = SentenceTransformerTrainingArguments(
    output_dir="/content/ckpt", num_train_epochs=2, per_device_train_batch_size=64, learning_rate=2e-5,
    warmup_ratio=0.1, fp16=True, batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch", save_strategy="no", logging_steps=20, seed=SEED, report_to="none")
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=Dataset.from_list(triplas),
                                     loss=losses.MultipleNegativesRankingLoss(model), evaluator=ir)
trainer.train()
model.save(str(MODELO_DIR))

## 11. Avaliação no conjunto rotulado

`avaliar.main` compara com o `baseline.json` do repositório (v1, índice sem GL*).

In [ ]:
m_novo = avaliar_modelo(model)
b = json.loads(avaliar.BASELINE.read_text(encoding="utf-8"))
criterios = {
    "recall@3 >= baseline + 0.10": m_novo["recall@3"] >= b["recall@3"] + 0.10,
    "ruido@3 <= baseline - 0.10": m_novo["ruido@3"] <= b["ruido@3"] - 0.10,
    "gap > baseline": m_novo["gap"] > b["gap"],
}
for k, ok in criterios.items():
    print(("OK   " if ok else "FALHA"), k)

# ruído@3 e vazio correto dependem do corte; o modelo novo muda a escala dos scores
casos = [json.loads(l) for l in avaliar.CASOS.read_text(encoding="utf-8").splitlines() if l.strip()]
res = [[(d["titulo"], d["score"]) for d in rag.buscar(c["relato"], c["especie"], k=10)] for c in casos]
print("\ncorte  ruido@3  vazio_correto")
for t in np.arange(0.78, 0.95, 0.01):
    m = avaliar.metricas(casos, res, t)
    print(f"{t:.2f}   {m['ruido@3']:.3f}    {m['vazio_correto']:.3f}")

## 12. Exportação ONNX int8 (layout do Xenova)

O `Embedder` do backend passa `token_type_ids`; a célula avisa se o modelo exportado não declara esse input (o `Embedder` precisa passar só os inputs da sessão).

In [ ]:
import shutil

import onnxruntime as ort
from google.colab import files
from optimum.onnxruntime import ORTModelForFeatureExtraction, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from tokenizers import Tokenizer

tmp = Path("/content/onnx_fp32")
ORTModelForFeatureExtraction.from_pretrained(str(MODELO_DIR), export=True).save_pretrained(tmp)
ORTQuantizer.from_pretrained(tmp).quantize(save_dir=tmp / "q",
                                           quantization_config=AutoQuantizationConfig.avx2(is_static=False))
(ONNX_DIR / "onnx").mkdir(parents=True, exist_ok=True)
shutil.copy(tmp / "q" / "model_quantized.onnx", ONNX_DIR / "onnx" / "model_quantized.onnx")
for nome in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "config.json"]:
    if (MODELO_DIR / nome).exists():
        shutil.copy(MODELO_DIR / nome, ONNX_DIR / nome)

sess = ort.InferenceSession(str(ONNX_DIR / "onnx" / "model_quantized.onnx"), providers=["CPUExecutionProvider"])
entradas = [i.name for i in sess.get_inputs()]
print("inputs:", entradas)
if "token_type_ids" not in entradas:
    print("AVISO: sem token_type_ids — Embedder.embed deve passar só os inputs da sessão")

tok = Tokenizer.from_file(str(ONNX_DIR / "tokenizer.json"))
tok.enable_truncation(512)


def onnx_embed(textos):
    """Um texto por vez, como o retriever; mean pooling + L2."""
    out = []
    for t in textos:
        e = tok.encode(t)
        ids, mask = np.array([e.ids]), np.array([e.attention_mask])
        feed = {"input_ids": ids, "attention_mask": mask, "token_type_ids": np.zeros_like(ids)}
        h = sess.run(None, {k: v for k, v in feed.items() if k in entradas})[0]
        v = (h * mask[..., None]).sum(1) / mask.sum(1, keepdims=True)
        out.append(v / np.linalg.norm(v, axis=1, keepdims=True))
    return np.concatenate(out)


amostra = [passage(c) for c in CHUNKS[::750]] + [query(p) for p in val[:10]]
cos = (onnx_embed(amostra) * model.encode(amostra, normalize_embeddings=True)).sum(1)
assert cos.min() > 0.99, cos
print(f"cosseno ONNX int8 x PyTorch: min {cos.min():.4f}")

shutil.make_archive("/content/arkive-e5-vet-onnx", "zip", ONNX_DIR)
files.download("/content/arkive-e5-vet-onnx.zip")

## 13. Publicação no HF Hub (opcional)

Preencha `HUB_REPO` na célula 3. Usa o secret `HF_TOKEN`.

In [ ]:
if HUB_REPO:
    from huggingface_hub import HfApi

    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HUB_REPO, exist_ok=True)
    api.upload_folder(folder_path=str(ONNX_DIR), repo_id=HUB_REPO)
    print(f"https://huggingface.co/{HUB_REPO}")